
# SpatialShiftAI - Retrain Model 2 & Model 3 (Google Colab, cloud GPU)

This notebook trains the two missing SpatialShiftAI models using the **existing**
repository training code and the contracts defined in `training/SpatialShiftAI.ipynb`:

| | Model 2 | Model 3 |
|---|---|---|
| Task | building footprint segmentation | bitemporal change detection |
| Dataset | WHU Building Dataset | LEVIR-CD (`ericyu/LEVIRCD_Cropped_256`) |
| Architecture | torchvision DeepLabV3-ResNet50, 1-channel head | Siamese ResNet18 + abs-diff + decoder, 1-channel head |
| Input | RGB 256x256, scaled to [0,1], no mean/std | RGB 256x256, ImageNet normalisation |
| Loss | `0.5*BCEWithLogits + 0.5*Dice` | `0.5*BCEWithLogits + 0.5*Dice` |
| Optimizer | AdamW lr=1e-4 wd=1e-4 | AdamW lr=1e-4 wd=1e-4, grad-norm clip 1.0 |
| Epochs | 12 | 5 |
| Best model | lowest validation loss | highest validation IoU |
| Output | `building_deeplabv3_resnet50_best.pt` | `siamese_resnet18_change_best.pt` |

## How to run

1. **Runtime -> Change runtime type -> GPU** (pick a T4). This is required.
2. Click **Runtime -> Run all**.
3. When prompted, **authorise Google Drive** (click "Allow").
4. **Leave the Colab tab open** until the final report prints. Colab deletes
   `/content` on reset, so every checkpoint/metric/history/card is written to
   `/content/drive/MyDrive/SpatialShiftAI_trained_models/`.
5. **Verify the final report** - it prints size, SHA256, reload test and a real
   inference test per model. Training is reported successful only when the real
   `.pt` files exist, reload, and produce correct-shaped masks.

> The previous Model 2 Colab run failed during dataset download. `train.py` is
> now hardened with a resumable retried download, explicit dataset verification
> and GPU sanity checks that run *before* full training.


In [ ]:
# CELL 1 - GPU GATE. Training NEVER falls back to CPU; this stops immediately.
import sys

import torch

print("torch      :", torch.__version__)
print("CUDA build :", torch.version.cuda)
print("CUDA avail :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit(
        "\nSTOPPED: no CUDA GPU detected.\n"
        "Go to Runtime -> Change runtime type -> GPU (T4), then Run all again.\n"
        "Model 2/3 training is GPU-only by design and will not silently "
        "train on CPU."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
CUDA_VERSION = torch.version.cuda
print("GPU NAME   :", GPU_NAME)
print("GPU memory : %.1f GiB" % GPU_MEM_GB)
print("GPU OK - training may proceed.\n")

In [ ]:
# CELL 2 - Mount Google Drive and create the artifact directory.
# Nothing important is left only under /content.
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SpatialShiftAI_trained_models")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

M2_DIR = DRIVE_ROOT / "models" / "model2_building_extractor"
M3_DIR = DRIVE_ROOT / "models" / "model3_change_detection"
for directory in (M2_DIR, M3_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Drive root  :", DRIVE_ROOT)
print("M2 artifacts:", M2_DIR)
print("M3 artifacts:", M3_DIR)

In [ ]:
# CELL 3 - Fetch the repository and install missing dependencies.
# torch/torchvision are NOT reinstalled: Colab's CUDA build is already correct.
import importlib.util
import subprocess

REPO_URL = "https://github.com/shrijit2609/GeoAI.git"
REPO_DIR = Path("/content/SpatialShiftAI")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", "main", REPO_URL, str(REPO_DIR)], check=True
    )

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("repo commit:", commit)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "opencv-python-headless", "rasterio", "scikit-learn", "pandas",
     "huggingface_hub", "pyarrow", "pillow"],
    check=True,
)
print("dependencies installed")


def load_module(path: Path, name: str):
    """Import one of the repository training scripts as a module."""
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


m2_train = load_module(REPO_DIR / "training/model2_building_extractor/train.py", "m2_train")
m3_train = load_module(REPO_DIR / "training/model3_change_detection/train.py", "m3_train")
print("loaded M2/M3 training modules from", REPO_DIR)

# --- API contract guard -------------------------------------------------
# The notebook is cloned fresh from origin/main. If the repository is older
# than the hardened trainers, the required helpers are missing and training
# cannot run. Fail loudly here with an actionable message instead of an
# AttributeError several cells later.
REQUIRED_API = {
    "m2_train": ["download_dataset", "verify_dataset", "discover_pairs",
                 "BuildingDataset", "build_model", "combined_loss", "train",
                 "TRAIN_BATCH", "EPOCHS", "IMAGE_SIZE"],
    "m3_train": ["download_dataset", "verify_dataset", "load_splits",
                 "ChangeDataset", "SiameseChangeNet", "combined_loss", "train",
                 "detect_columns", "extract_row", "EPOCHS", "IMAGE_SIZE"],
}
missing = []
for module_name, names in REQUIRED_API.items():
    module = sys.modules[module_name]
    for name in names:
        if not hasattr(module, name):
            missing.append(f"{module_name}.{name}")

import inspect

if "resume" not in inspect.signature(m2_train.train).parameters:
    missing.append("m2_train.train(resume=...)")

if missing:
    raise SystemExit(
        "\nSTOPPED: the cloned repository is missing the hardened training API.\n"
        f"cloned commit : {commit}\n"
        f"missing       : {', '.join(missing)}\n"
        "The retraining helpers (download_dataset/verify_dataset/resume) must be "
        "present on origin/main. Update the repository and re-run this notebook:\n"
        "  !rm -rf /content/SpatialShiftAI"
    )

print("training API verified: download_dataset, verify_dataset, resume present")
print(f"  m2: epochs={m2_train.EPOCHS} image={m2_train.IMAGE_SIZE} batch={m2_train.TRAIN_BATCH}")
print(f"  m3: epochs={m3_train.EPOCHS} image={m3_train.IMAGE_SIZE}")

In [ ]:
# CELL 4 - Download the dataset mirrors (retried + resumable) into the Colab
# VM's fast local disk. Only trained weights are written to Drive.
M2_SCRIPT = REPO_DIR / "training/model2_building_extractor/train.py"
M3_SCRIPT = REPO_DIR / "training/model3_change_detection/train.py"
M2_DATA = Path("/content/datasets/whu_building")
M3_DATA = Path("/content/datasets/levir_cd")

m2_train.download_dataset(M2_DATA)
m3_train.download_dataset(M3_DATA)
print("WHU mirror ready at", M2_DATA)
print("LEVIR mirror ready at", M3_DATA)

In [ ]:
# CELL 5 - MODEL 2 GPU SANITY TEST (runs before any real training).
# dataset -> one batch -> CUDA -> forward -> loss -> backward -> optimizer step.
import torch

torch.manual_seed(0)
device = torch.device("cuda")

train_df, val_df, test_df = m2_train.discover_pairs(M2_DATA)
print(f"M2 dataset: train={len(train_df)} val={len(val_df)} test={len(test_df)}")
assert len(train_df) > 0

dataset = m2_train.BuildingDataset(train_df, training=True)
batch = torch.utils.data.DataLoader(
    dataset, batch_size=m2_train.TRAIN_BATCH, shuffle=True, num_workers=2
)
images, masks = next(iter(batch))
images, masks = images.to(device), masks.to(device)
print("batch on CUDA:", tuple(images.shape), tuple(masks.shape))
assert images.shape[1] == 3 and masks.shape[1] == 1

model = m2_train.build_model(pretrained=True).to(device)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
optimizer.zero_grad(set_to_none=True)
logits = model(images)["out"]
loss = m2_train.combined_loss(logits, masks)
loss.backward()
optimizer.step()
print(f"M2 sanity: logits={tuple(logits.shape)} loss={float(loss):.5f}")
assert logits.shape[1] == 1 and torch.isfinite(loss)
print("M2 GPU SANITY TEST PASSED\n")

del model, optimizer
torch.cuda.empty_cache()

In [ ]:
# CELL 6 - MODEL 3 GPU SANITY TEST (runs before any real training).
train_rows, val_rows, test_rows = m3_train.load_splits(M3_DATA)
print(f"M3 dataset: train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}")
assert len(train_rows) > 0

batch3 = torch.utils.data.DataLoader(
    m3_train.ChangeDataset(train_rows, augment=True), batch_size=4, shuffle=True, num_workers=2
)
before, after, target = next(iter(batch3))
before, after, target = before.to(device), after.to(device), target.to(device)
print("batch on CUDA:", tuple(before.shape), tuple(after.shape), tuple(target.shape))

model3 = m3_train.SiameseChangeNet(pretrained=True).to(device)
model3.train()
optimizer3 = torch.optim.AdamW(model3.parameters(), lr=1e-4, weight_decay=1e-4)
optimizer3.zero_grad(set_to_none=True)
out = model3(before, after)
loss3 = m3_train.combined_loss(out, target)
loss3.backward()
torch.nn.utils.clip_grad_norm_(model3.parameters(), 1.0)
optimizer3.step()
print(f"M3 sanity: out={tuple(out.shape)} loss={float(loss3):.5f}")
assert out.shape[1] == 1 and torch.isfinite(loss3)
print("M3 GPU SANITY TEST PASSED\n")

del model3, optimizer3
torch.cuda.empty_cache()

In [ ]:
# CELL 7 - TRAIN MODEL 2 (real, full 12-epoch WHU run, writing to Drive).
# Uses the repository trainer unchanged so the contract is preserved.
# Re-running resumes from the best checkpoint already on Drive.
import json

M2_BEST = M2_DIR / "building_deeplabv3_resnet50_best.pt"

print("training Model 2 ->", M2_BEST)
print("epochs:", m2_train.EPOCHS, "| image size:", m2_train.IMAGE_SIZE,
      "| batch:", m2_train.TRAIN_BATCH)

m2_metrics = m2_train.train(M2_DATA, DRIVE_ROOT, resume=M2_BEST.exists())
print(json.dumps(m2_metrics, indent=2))

if not M2_BEST.is_file():
    raise SystemExit("Model 2 training finished but no checkpoint was written.")
print("M2 checkpoint present on Drive:", M2_BEST)

In [ ]:
# CELL 8 - VERIFY MODEL 2 through the PRODUCTION backend adapter.
# exists / size / SHA256 / strict reload / real inference / output shape.
import hashlib


def sha256_of(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


REPORT: dict[str, dict] = {}

sys.path.insert(0, str(REPO_DIR / "backend"))
from app.ml.common import ModelStatus
from app.ml.model_registry import ModelRegistry

M2_SIZE = M2_BEST.stat().st_size
M2_SHA = sha256_of(M2_BEST)
print("M2 checkpoint :", M2_BEST)
print("M2 size       :", f"{M2_SIZE:,} bytes")
print("M2 sha256     :", M2_SHA)

raw2 = torch.load(M2_BEST, map_location="cpu", weights_only=False)
assert "model_state_dict" in raw2, "checkpoint has no model_state_dict"
print("M2 state_dict tensors:", len(raw2["model_state_dict"]))
print("M2 best epoch :", raw2.get("epoch"))

registry2 = ModelRegistry(DRIVE_ROOT / "models", device="cpu")
health2 = registry2.health("building_extractor")
print("M2 adapter status:", health2.status.value, "| error:", health2.error)
assert health2.status is ModelStatus.READY, health2.error

sample = Path(str(test_df.iloc[0].image))
probability, binary, _ = registry2.get("building_extractor").predict_masks(str(sample))
print("M2 real inference on", sample.name, "-> mask", list(binary.shape),
      "| building px %", round(float(binary.mean() * 100), 3))
assert binary.shape == (256, 256), f"unexpected mask shape {binary.shape}"

REPORT["M2"] = {
    "status": health2.status.value,
    "checkpoint": str(M2_BEST),
    "size": M2_SIZE,
    "sha256": M2_SHA,
    "metrics": m2_metrics,
    "inference_mask_shape": list(binary.shape),
    "load_test": "PASS",
    "inference_test": "PASS",
}
print("M2 VERIFICATION PASSED\n")

In [ ]:
# CELL 9 - TRAIN MODEL 3 (real, full 5-epoch LEVIR run, writing to Drive).
M3_BEST = M3_DIR / "siamese_resnet18_change_best.pt"

print("training Model 3 ->", M3_BEST)
print("epochs:", m3_train.EPOCHS, "| image size:", m3_train.IMAGE_SIZE,
      "| grad clip: 1.0")

m3_metrics = m3_train.train(M3_DATA, DRIVE_ROOT)
print(json.dumps(m3_metrics, indent=2))

if not M3_BEST.is_file():
    raise SystemExit("Model 3 training finished but no checkpoint was written.")
print("M3 checkpoint present on Drive:", M3_BEST)

In [ ]:
# CELL 10 - VERIFY MODEL 3 through the PRODUCTION backend adapter.
import pandas as pd

M3_SIZE = M3_BEST.stat().st_size
M3_SHA = sha256_of(M3_BEST)
print("M3 checkpoint :", M3_BEST)
print("M3 size       :", f"{M3_SIZE:,} bytes")
print("M3 sha256     :", M3_SHA)

raw3 = torch.load(M3_BEST, map_location="cpu", weights_only=False)
assert "model_state_dict" in raw3, "checkpoint has no model_state_dict"
print("M3 state_dict tensors:", len(raw3["model_state_dict"]))
print("M3 best epoch :", raw3.get("epoch"))

registry3 = ModelRegistry(DRIVE_ROOT / "models", device="cpu")
health3 = registry3.health("change_detector")
print("M3 adapter status:", health3.status.value, "| error:", health3.error)
assert health3.status is ModelStatus.READY, health3.error

frame = pd.read_parquet(
    M3_DATA / "data" / "test-00000-of-00001-31d7c3e3444e5b5d.parquet", engine="pyarrow"
)
columns3 = m3_train.detect_columns(frame)
before_np, after_np, _mask_np = m3_train.extract_row(frame.iloc[0], columns3)

result3 = registry3.get("change_detector").predict(before_np, after_np)
print("M3 real inference -> decision:", result3.decision,
      "| mask", result3.evidence["mask_shape"],
      "| changed px %", round(result3.evidence["changed_pixel_percentage"], 3))
assert result3.evidence["mask_shape"] == [256, 256], "unexpected change-mask shape"

REPORT["M3"] = {
    "status": health3.status.value,
    "checkpoint": str(M3_BEST),
    "size": M3_SIZE,
    "sha256": M3_SHA,
    "metrics": m3_metrics,
    "inference_mask_shape": result3.evidence["mask_shape"],
    "decision": result3.decision,
    "load_test": "PASS",
    "inference_test": "PASS",
}
print("M3 VERIFICATION PASSED\n")

In [ ]:
# CELL 11 - FINAL COMPLETION REPORT.
# Only reached when both real checkpoints exist, reload and infer correctly.
print("=" * 72)
print("SpatialShiftAI - Model 2 & Model 3 retraining COMPLETE")
print("=" * 72)
print(f"GPU        : {GPU_NAME} ({GPU_MEM_GB:.1f} GiB)")
print(f"CUDA       : {CUDA_VERSION}")

for key in ("M2", "M3"):
    entry = REPORT[key]
    metrics = entry["metrics"]
    best = metrics.get("best_validation_loss", metrics.get("best_validation_iou"))
    print(f"\n{key}:")
    print(f"  STATUS          : {entry['status']}")
    print(f"  CHECKPOINT      : {entry['checkpoint']}")
    print(f"  SIZE            : {entry['size']:,} bytes")
    print(f"  SHA256          : {entry['sha256']}")
    print(f"  BEST VAL METRIC : {best}")
    print(f"  BEST EPOCH      : {metrics.get('best_epoch')}")
    print(f"  TEST METRICS    : {metrics.get('test_metrics')}")
    print(f"  LOAD TEST       : {entry['load_test']}")
    print(f"  INFERENCE TEST  : {entry['inference_test']} "
          f"(mask {entry['inference_mask_shape']})")

print("\nDRIVE OUTPUT:")
for directory in sorted(DRIVE_ROOT.rglob("*")):
    if directory.is_file():
        print(f"  {directory.relative_to(DRIVE_ROOT)}  ({directory.stat().st_size:,} bytes)")

print("\n" + json.dumps(REPORT, indent=2))
print("\nBoth real checkpoints are on Google Drive and reload through the "
      "production backend adapters. They can now be integrated into the "
      "deployed SpatialShiftAI service.")